# Detector multiclase de EPP con YOLO

Este notebook entrena y evalua un detector de objetos para EPP/PPE en obras de construccion. Esta preparado para ejecutarse tanto localmente como en Google Colab.

La salida principal no es una etiqueta binaria por imagen, sino detecciones multiclase y un reporte de cumplimiento por persona.

Flujo:

- Usar solo `css-data`, que contiene `train`, `valid` y `test` en formato YOLO.
- Entrenar YOLO sobre las 10 clases del dataset.
- Evaluar con metricas de deteccion como mAP, precision y recall.
- Convertir detecciones en un reporte operativo de cumplimiento.


## 1. Verificacion del entorno

En Colab, esta seccion instala las dependencias necesarias y usa GPU CUDA si esta disponible. Para guardar pesos entre sesiones, activa `USE_GOOGLE_DRIVE = True`.

Localmente, usa el entorno `.venv` del proyecto y guarda resultados en `outputs/`.


In [ ]:
from pathlib import Path
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = False

if IN_COLAB:
    packages = ["ultralytics", "kagglehub", "pyyaml"]
    missing = [pkg for pkg in packages if importlib.util.find_spec(pkg.split("==")[0].replace("pyyaml", "yaml")) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

PROJECT_ROOT = Path.cwd()

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/construction-safety-ppe")
else:
    OUTPUT_BASE = PROJECT_ROOT / "outputs"

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
YOLO_CONFIG_DIR = OUTPUT_BASE / ".ultralytics"
YOLO_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(YOLO_CONFIG_DIR.resolve())

import torch
import ultralytics
from ultralytics import YOLO

print(f"Python: {sys.version.split()[0]} ({platform.system()} {platform.machine()})")
print(f"Colab: {IN_COLAB}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Output base: {OUTPUT_BASE}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Torch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Preparacion del dataset YOLO

KaggleHub descarga una carpeta que incluye datos, archivos fuente y resultados previos. Para entrenar y evaluar correctamente se usa solo `css-data`.

In [ ]:
from pathlib import Path
import yaml
import kagglehub
import pandas as pd

DATASET_HANDLE = "snehilsanyal/construction-site-safety-image-dataset-roboflow"
DOWNLOAD_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
DATASET_ROOT = DOWNLOAD_ROOT / "css-data"

CLASS_NAMES = [
    "Hardhat",
    "Mask",
    "NO-Hardhat",
    "NO-Mask",
    "NO-Safety Vest",
    "Person",
    "Safety Cone",
    "Safety Vest",
    "machinery",
    "vehicle",
]

for split in ["train", "valid", "test"]:
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f"No existe la estructura YOLO esperada para {split}: {image_dir} / {label_dir}")

OUTPUT_DIR = OUTPUT_BASE / "yolo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML_PATH = OUTPUT_DIR / "ppe_data.yaml"

yolo_data = {
    "path": str(DATASET_ROOT.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": CLASS_NAMES,
}
DATA_YAML_PATH.write_text(yaml.safe_dump(yolo_data, sort_keys=False), encoding="utf-8")

print("Download root:", DOWNLOAD_ROOT)
print("Dataset root usado:", DATASET_ROOT)
print("YOLO data config:", DATA_YAML_PATH)
print(DATA_YAML_PATH.read_text(encoding="utf-8"))

## 3. Auditoria especifica para deteccion

Antes de entrenar, valida conteos por split, correspondencia imagen-etiqueta y balance de clases por anotacion.

In [ ]:
from collections import Counter

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

split_rows = []
annotation_counter = Counter()
missing_label_files = []
empty_label_files = []
invalid_class_ids = []

for split in ["train", "valid", "test"]:
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"
    image_paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    label_paths = sorted(label_dir.glob("*.txt"))

    for image_path in image_paths:
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            missing_label_files.append(str(image_path.relative_to(DATASET_ROOT)))
            continue
        lines = [line.strip() for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
        if not lines:
            empty_label_files.append(str(label_path.relative_to(DATASET_ROOT)))
        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            try:
                class_id = int(float(parts[0]))
            except (IndexError, ValueError):
                invalid_class_ids.append((str(label_path.relative_to(DATASET_ROOT)), line_number, line))
                continue
            if class_id < 0 or class_id >= len(CLASS_NAMES):
                invalid_class_ids.append((str(label_path.relative_to(DATASET_ROOT)), line_number, line))
                continue
            annotation_counter[CLASS_NAMES[class_id]] += 1

    split_rows.append(
        {
            "split": split,
            "images": len(image_paths),
            "labels": len(label_paths),
        }
    )

split_df = pd.DataFrame(split_rows)
class_balance_df = pd.DataFrame(
    [{"class_name": name, "annotations": annotation_counter[name]} for name in CLASS_NAMES]
).sort_values("annotations", ascending=False)

print("Conteo por split:")
display(split_df)
print("Balance por clase:")
display(class_balance_df)
print("Imagenes sin label:", len(missing_label_files))
print("Labels vacios:", len(empty_label_files))
print("Lineas con class_id invalido:", len(invalid_class_ids))

## 4. Entrenamiento YOLO multiclase

Este es el baseline de deteccion. En CPU puede tardar; para una primera prueba baja `EPOCHS` a 5. Para un resultado mas serio usa GPU y sube `EPOCHS` a 50-100.

In [ ]:
MODEL_WEIGHTS = "yolov8n.pt"
EPOCHS = 30
IMG_SIZE = 640
BATCH_SIZE = 8
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_TRAINING = True

if RUN_TRAINING:
    model = YOLO(MODEL_WEIGHTS)
    train_results = model.train(
        data=str(DATA_YAML_PATH),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=str((OUTPUT_BASE / "yolo_runs").resolve()),
        name="ppe_yolov8n_baseline",
        exist_ok=True,
        patience=10,
        seed=42,
    )
else:
    print("RUN_TRAINING=False. Cambialo a True para entrenar.")

## 5. Evaluacion en validacion y test

Para seguridad, mira especialmente el recall de las clases `NO-Hardhat`, `NO-Mask` y `NO-Safety Vest`. Esas clases representan condiciones inseguras.

In [ ]:
RUN_EVALUATION = True

if RUN_EVALUATION:
    best_model_path = OUTPUT_BASE / "yolo_runs" / "ppe_yolov8n_baseline" / "weights" / "best.pt"
    eval_model = YOLO(str(best_model_path if best_model_path.exists() else MODEL_WEIGHTS))

    print("Validacion:")
    val_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="val", imgsz=IMG_SIZE, device=DEVICE)

    print("Test:")
    test_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="test", imgsz=IMG_SIZE, device=DEVICE)
else:
    print("RUN_EVALUATION=False. Cambialo a True para evaluar.")

## 6. Prediccion y reporte de cumplimiento

Esta seccion convierte detecciones en un reporte por persona. Las clases `NO-*` marcan incumplimientos; las cajas se asocian con una persona cuando el centro de la caja cae dentro del bounding box de esa persona.

In [ ]:
import sys
from dataclasses import asdict, dataclass
from pprint import pprint
from typing import Iterable

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

try:
    from src.ppe_compliance import Detection, generate_ppe_compliance_report
except ModuleNotFoundError:
    @dataclass(frozen=True)
    class Detection:
        class_name: str
        confidence: float
        box_xyxy: tuple[float, float, float, float]

    def _center_inside(item_box, container_box):
        x1, y1, x2, y2 = item_box
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        left, top, right, bottom = container_box
        return left <= cx <= right and top <= cy <= bottom

    def generate_ppe_compliance_report(
        detections: Iterable[Detection],
        *,
        person_class: str = "Person",
        ppe_classes: tuple[str, ...] = ("Hardhat", "Mask", "Safety Vest"),
        violation_classes: tuple[str, ...] = ("NO-Hardhat", "NO-Mask", "NO-Safety Vest"),
        min_confidence: float = 0.25,
    ):
        filtered = [det for det in detections if det.confidence >= min_confidence]
        persons = [det for det in filtered if det.class_name == person_class]
        ppe_items = [det for det in filtered if det.class_name in ppe_classes]
        violations = [det for det in filtered if det.class_name in violation_classes]

        person_reports = []
        assigned_violation_ids = set()
        for person_index, person in enumerate(persons, start=1):
            person_ppe = tuple(sorted({item.class_name for item in ppe_items if _center_inside(item.box_xyxy, person.box_xyxy)}))
            person_violations = tuple(sorted({violation.class_name for violation in violations if _center_inside(violation.box_xyxy, person.box_xyxy)}))
            for violation_id, violation in enumerate(violations):
                if violation.class_name in person_violations and _center_inside(violation.box_xyxy, person.box_xyxy):
                    assigned_violation_ids.add(violation_id)
            person_reports.append(
                {
                    "person_index": person_index,
                    "person_box_xyxy": person.box_xyxy,
                    "detected_ppe": person_ppe,
                    "violations": person_violations,
                    "status": "inseguro" if person_violations else "requiere_revision",
                }
            )

        unassigned_violations = tuple(sorted(violations[index].class_name for index in range(len(violations)) if index not in assigned_violation_ids))
        violation_count = sum(len(person["violations"]) for person in person_reports) + len(unassigned_violations)
        return {
            "status": "inseguro" if violation_count else "sin_violaciones_detectadas",
            "person_count": len(person_reports),
            "violation_count": violation_count,
            "persons": person_reports,
            "unassigned_violations": unassigned_violations,
        }

sample_image = next((DATASET_ROOT / "test" / "images").glob("*.jpg"))
model_path = OUTPUT_BASE / "yolo_runs" / "ppe_yolov8n_baseline" / "weights" / "best.pt"
predict_model = YOLO(str(model_path if model_path.exists() else MODEL_WEIGHTS))

results = predict_model.predict(
    source=str(sample_image),
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str((OUTPUT_BASE / "predictions").resolve()),
    name="sample_compliance",
    exist_ok=True,
)

result = results[0]
detections = []
for box in result.boxes:
    class_id = int(box.cls.item())
    detections.append(
        Detection(
            class_name=result.names[class_id],
            confidence=float(box.conf.item()),
            box_xyxy=tuple(float(value) for value in box.xyxy[0].tolist()),
        )
    )

report = generate_ppe_compliance_report(detections)
print("Imagen:", sample_image)
print("Detecciones:")
pprint([asdict(det) for det in detections])
print("Reporte de cumplimiento:")
if hasattr(report, "__dataclass_fields__"):
    pprint(asdict(report))
else:
    pprint(report)


## 7. Siguientes mejoras

- Revisar visualmente falsos negativos de `NO-Hardhat`, `NO-Mask` y `NO-Safety Vest`.
- Ajustar `conf` por clase para priorizar recall en condiciones inseguras.
- Probar `yolov8s.pt` o un modelo mas reciente si se entrena con GPU.
- Validar con imagenes reales de una obra distinta al dataset.
- Mejorar la asociacion persona-EPP usando IoU, distancia relativa por region corporal o tracking en video.